# Stage 4 — Model training and OOF prediction

This notebook is the single Stage-4 orchestration entry point. The reusable implementations live in `src/models/`; the notebook loads protected artifacts, runs each model, and displays aggregate diagnostics.

> **PhysioNet DUA:** run only in the controlled Kaggle/Colab environment containing the protected Stage-2/3 artifacts. Never display patient rows or identifiers. OOF predictions and fitted models must remain in the gitignored `data/` and `results/` directories.

## Stage-4 protocol

1. Reuse the frozen 20% internal holdout and five development folds from Stage 3.
2. Fit imputation and numeric scaling on original training-fold rows, then run SMOTENC, then one-hot encode categories. Never refit a scaler on synthetic rows. Class weights are also training-fold only.
3. Select hyperparameters by mean development-fold AUROC; use mean AUPRC as the tie-breaker.
4. Produce exactly one out-of-fold probability for every development row.
5. Refit the selected configuration on the complete development set.
6. Do not fit, predict, evaluate, or tune against the internal test partition in this stage.
7. Leave F1-threshold selection and final model comparison to Stage 5.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import N_CV_FOLDS, RANDOM_SEED
from src.models.classic import (
    logistic_regression_candidates,
    load_static_stage4_inputs,
    save_static_training_result,
    train_logistic_regression,
    train_xgboost,
    xgboost_candidates,
)
from src.resampling_audit import audit_smotenc_training_fold

# Version the corrected run so the previous screening artifacts remain intact.
ARTIFACT_SUFFIX = 'smotenc_scaled_v2'

np.random.seed(RANDOM_SEED)
print(f'Project root: {ROOT}')
print(f'Frozen CV folds: {N_CV_FOLDS}; random seed: {RANDOM_SEED}')

## 1. Load and revalidate protected Stage-2/3 artifacts

The loader checks row alignment, frozen assignments, all five folds, and patient separation. This cell prints aggregate counts only.

In [ ]:
static, splits = load_static_stage4_inputs()
assignments = splits.assignments
dev = assignments['split'].eq('dev')
test = assignments['split'].eq('test')

assert len(splits.dev_indices) + len(splits.test_indices) == len(static)
assert set(assignments.loc[dev, 'cv_fold']) == set(range(N_CV_FOLDS))
assert set(assignments.loc[dev, 'subject_id']).isdisjoint(
    set(assignments.loc[test, 'subject_id'])
)

print(f'Total rows: {len(static):,}')
print(f'Development rows: {dev.sum():,}')
print(f'Internal-test rows (sealed): {test.sum():,}')
print(f'Development prevalence: {assignments.loc[dev, "label"].mean():.3%}')

### SMOTENC correction and optional quality audit

Restart the kernel after updating the source, then rerun this notebook with the existing Stage-2 features and frozen Stage-3 assignments; do not regenerate the splits. Both LR and XGBoost now use training-only numeric scaling before SMOTENC. Even `scale_numeric=False` cannot disable this distance-scaling requirement. Validation/prediction uses that same fitted scaler and never resamples.

The optional audit below separately resamples only the first fold's training partition and prints aggregate synthetic-quality counts. It does not train a model, touch validation/test rows, or change the screening. Check the reported denominators: missing features mean a check is unavailable, not passed. Fractional counts describe feature-space interpolation; contradictory missing flags/counts are a separate warning. No counts are rounded and no missing flags are silently repaired. This is a spot check, not proof that every synthetic sample or fold is clinically plausible.

The corrected run uses the `smotenc_scaled_v2` artifact suffix (plus `_smoke` for smoke runs), preserving the old unsuffixed and `_smoke` artifacts. Repeating the same version still replaces its own artifacts.

In [ ]:
RUN_SMOTENC_QUALITY_AUDIT = False

if RUN_SMOTENC_QUALITY_AUDIT:
    audit_train_indices, _ = next(splits.iter_cv())
    audit_rows = []
    for ratio in (0.10, 0.25, 0.50, 1.00):
        audit_rows.append({
            'fold': 0,
            'sampling_strategy': ratio,
            **audit_smotenc_training_fold(
                static.iloc[audit_train_indices],
                sampling_strategy=ratio,
                random_state=RANDOM_SEED,
            ),
        })
    display(pd.DataFrame(audit_rows))
else:
    print('Optional training-fold quality audit is off; set RUN_SMOTENC_QUALITY_AUDIT=True to run.')

## 2. Inspect the imbalance-screening space

The smoke profile has three configurations and validates baseline, fold-derived class weighting, and SMOTENC on all five folds. The screening profile fixes LR at `C=1`, L2 and compares six mutually exclusive imbalance strategies, isolating the effect of imbalance handling. SMOTENC targets minority/majority ratios of 0.10, 0.25, 0.50, or 1.00 and runs before one-hot encoding; it is never combined with class weighting. Full LR hyperparameter tuning is intentionally deferred until the strategy shortlist is fixed.

In [ ]:
lr_smoke_candidates = logistic_regression_candidates(profile='smoke')
lr_screening_candidates = logistic_regression_candidates(profile='screening')
print(f'LR Smoke candidates: {len(lr_smoke_candidates)}')
print(f'LR Imbalance-screening candidates: {len(lr_screening_candidates)}')
display(pd.DataFrame([candidate.as_dict() for candidate in lr_screening_candidates]))

In [ ]:
xgb_smoke_candidates = xgboost_candidates(profile="smoke")
xgb_screening_candidates = xgboost_candidates(profile="screening")
print(f'XGBoost Smoke candidates: {len(xgb_smoke_candidates)}')
print(f'XGBoost Imbalance-screening candidates: {len(xgb_screening_candidates)}')
display(pd.DataFrame([candidate.as_dict() for candidate in xgb_screening_candidates]))

## 3. Smoke-run

This still uses all five frozen development folds, comparing baseline, fold-derived class weighting, and SMOTENC 0.25 for each model. LR uses `C=1`, L2. Smoke artifacts receive the version suffix plus `_smoke` and cannot overwrite the screening run.

In [ ]:
lr_smoke_result = train_logistic_regression(
    static,
    splits,
    profile='smoke',
    progress_callback=print,
)
lr_smoke_artifacts = save_static_training_result(
    lr_smoke_result, artifact_suffix=f'{ARTIFACT_SUFFIX}_smoke'
)
display(lr_smoke_result.strategy_metrics)
print(f'LR smoke best candidate: {lr_smoke_result.best_candidate.name}')
print(f'LR protected smoke OOF: {lr_smoke_artifacts.oof_path}')
print(f'LR protected per-strategy OOF: {lr_smoke_artifacts.strategy_oof_path}')

In [ ]:
xgb_smoke_result = train_xgboost(
    static,
    splits,
    profile="smoke",
    device="cuda",
    progress_callback=print,
)
xgb_smoke_artifacts = save_static_training_result(
    xgb_smoke_result, artifact_suffix=f'{ARTIFACT_SUFFIX}_smoke'
)
display(xgb_smoke_result.strategy_metrics)
print(f'XGBoost smoke best candidate: {xgb_smoke_result.best_candidate.name}')
print(f'XGBoost protected smoke OOF: {xgb_smoke_artifacts.oof_path}')
print(f'XGBoost protected per-strategy OOF: {xgb_smoke_artifacts.strategy_oof_path}')

## 4. Run the imbalance screening

Run this only after the smoke result completes and the assertions below remain clean. Change each model's gate to `True`; each screening performs 6 × 5 fitted pipelines across six strategies, then refits its winner on dev. Artifacts use `ARTIFACT_SUFFIX`, preserving the previous pipeline's results.

In [ ]:
RUN_LR_IMBALANCE_SCREENING = False

if RUN_LR_IMBALANCE_SCREENING:
    lr_result = train_logistic_regression(
        static,
        splits,
        profile='screening',
        progress_callback=print,
    )
    lr_artifacts = save_static_training_result(lr_result, artifact_suffix=ARTIFACT_SUFFIX)
    display(lr_result.strategy_metrics)
    print(f'Final LR candidate: {lr_result.best_candidate.name}')
    print(f'Protected LR OOF: {lr_artifacts.oof_path}')
    print(f'LR protected per-strategy OOF: {lr_artifacts.strategy_oof_path}')
    print(f'LR fitted development model: {lr_artifacts.model_path}')
else:
    print('LR screening is gated. Set RUN_LR_IMBALANCE_SCREENING=True when ready.')

In [ ]:
RUN_XGB_IMBALANCE_SCREENING = False

if RUN_XGB_IMBALANCE_SCREENING:
    xgb_result = train_xgboost(
        static,
        splits,
        profile="screening",
        device="cuda",
        progress_callback=print,
    )
    xgb_artifacts = save_static_training_result(xgb_result, artifact_suffix=ARTIFACT_SUFFIX)
    display(xgb_result.strategy_metrics)
    print(f'Final XGBoost candidate: {xgb_result.best_candidate.name}')
    print(f'Protected XGBoost OOF: {xgb_artifacts.oof_path}')
    print(f'XGBoost protected per-strategy OOF: {xgb_artifacts.strategy_oof_path}')
    print(f'XGBoost fitted development model: {xgb_artifacts.model_path}')

## 5. Leakage and completeness audit

The assertions apply to the final result when available, otherwise to the smoke run. They confirm that every development row has one finite OOF probability and that no internal-test row appears in the OOF artifact.

In [ ]:
checked_result = lr_result if RUN_LR_IMBALANCE_SCREENING else lr_smoke_result
oof = checked_result.oof_predictions

assert len(oof) == len(splits.dev_indices)
assert oof['row_index'].is_unique
assert set(oof['row_index']) == set(splits.dev_indices)
assert set(oof['row_index']).isdisjoint(splits.test_indices)
assert np.isfinite(oof['probability']).all()
assert oof['probability'].between(0, 1).all()
assert set(oof['cv_fold']) == set(range(N_CV_FOLDS))
strategy_oof = checked_result.strategy_oof_predictions
n_strategies = checked_result.strategy_metrics['strategy'].nunique()
assert len(strategy_oof) == len(splits.dev_indices) * n_strategies
assert strategy_oof.groupby('strategy')['row_index'].nunique().eq(len(splits.dev_indices)).all()
assert set(strategy_oof['row_index']).isdisjoint(splits.test_indices)

print(f'LR global-best OOF rows verified: {len(oof):,}')
print(f'LR per-strategy OOF rows verified: {len(strategy_oof):,}')
print('LR internal test remains sealed: no Stage-4 predictions were created for it.')

In [ ]:
checked_result = xgb_result if RUN_XGB_IMBALANCE_SCREENING else xgb_smoke_result
oof = checked_result.oof_predictions

assert len(oof) == len(splits.dev_indices)
assert oof['row_index'].is_unique
assert set(oof['row_index']) == set(splits.dev_indices)
assert set(oof['row_index']).isdisjoint(splits.test_indices)
assert np.isfinite(oof['probability']).all()
assert oof['probability'].between(0, 1).all()
assert set(oof['cv_fold']) == set(range(N_CV_FOLDS))
strategy_oof = checked_result.strategy_oof_predictions
n_strategies = checked_result.strategy_metrics['strategy'].nunique()
assert len(strategy_oof) == len(splits.dev_indices) * n_strategies
assert strategy_oof.groupby('strategy')['row_index'].nunique().eq(len(splits.dev_indices)).all()
assert set(strategy_oof['row_index']).isdisjoint(splits.test_indices)

print(f'XGBoost global-best OOF rows verified: {len(oof):,}')
print(f'XGBoost per-strategy OOF rows verified: {len(strategy_oof):,}')
print('XGBoost internal test remains sealed: no Stage-4 predictions were created for it.')

## How the common framework is reused

`train_static_model` owns the invariant workflow: validate frozen splits → compute fold-local weights or resample with SMOTENC → cross-fit every candidate → calculate fold AUROC/AUPRC/Brier → retain the best candidate and OOF predictions for every strategy → refit the global winner on the complete development set. A later model supplies only: (1) an estimator factory, (2) candidate configurations, and (3) preprocessing/imbalance flags.

LR and XGBoost screening are implemented. The next milestone is intentionally left open: add the class-weight/neighbor/ordinary-SMOTE sensitivity candidates, shortlist promising strategies, and then extend to RF/SVM/MLP/LSTM. SVM additionally needs fold-local probability calibration; XGBoost tuning may add fold-local early stopping while keeping SMOTENC/`scale_pos_weight` mutually exclusive; LSTM switches to the hourly tensor and loss-function ablation.

## LR milestone acceptance

Before implementing the remaining models, confirm only aggregate outputs: the six-row imbalance screening table, each strategy's best candidate, five-fold AUROC/AUPRC/Brier summaries, calibration intercept/slope, predicted-risk prevalence, development OOF row counts, and saved artifact paths. Do not share OOF files or patient-level rows. F1 threshold selection remains deferred to Stage 5.